# 03. Consumer PostgreSQL + pgvector
Notebook bazuje na wzorcu consumera Kafka->Postgres z laboratoriów, rozszerzonym o embedding tytułu [file:12].

In [ ]:
from kafka import KafkaConsumer
import psycopg2, json
from datetime import datetime
from sentence_transformers import SentenceTransformer

consumer = KafkaConsumer(
    "transactions",
    bootstrap_servers="kafka_streaming_lab:9092",
    auto_offset_reset="earliest",
    enable_auto_commit=True,
    group_id="transactions-pg-group",
    value_deserializer=lambda m: json.loads(m.decode("utf-8"))
)

conn = psycopg2.connect(
    dbname="mydb",
    user="myuser",
    password="myuserpass",
    host="postgres_bank_lab",
    port=5432
)
cursor = conn.cursor()
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

insert_sql = """
INSERT INTO transactions (
    sender, receiver, amount, transaction_timestamp,
    device_sender, device_receiver, title, title_embedding
) VALUES (%s, %s, %s, %s, %s, %s, %s, %s)
"""

print("Kafka-to-Postgres processor running...")

for msg in consumer:
    data = msg.value
    embedding = model.encode(data["title"]).tolist()
    tx_timestamp = datetime.fromisoformat(data["timestamp"].replace(" ", ""))

    cursor.execute(insert_sql, (
        data["sender"],
        data["receiver"],
        data["amount"],
        tx_timestamp,
        data["device_sender"],
        data["device_receiver"],
        data["title"],
        embedding
    ))
    conn.commit()
    print(f"Inserted into PostgreSQL: {data}")